# IssueFix-RL — SFT Training on Kaggle 2× T4

**Upload this notebook to Kaggle, enable 2× GPU accelerator, turn on Internet, then Save & Run All.**

Pipeline:
1. Clone repo
2. Install deps
3. Run training — auto-distributes across both T4s via `notebook_launcher`
4. Zip best checkpoint → `/kaggle/working/`
5. Push to Kaggle Models page

Best checkpoint + logs persist as a Kaggle notebook version (visible in Output tab).

---
**Before running:**
- Attach your training data as a Kaggle dataset (jsonl with `prompt`/`response` keys)
- Add your wandb API key as a Kaggle secret named `WANDB_API_KEY` (optional)
- Set `DATA_PATH` in cell 2 to your mounted dataset path

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
# !pip install flash-attn --no-build-isolation

In [ ]:
# ── EDIT THESE ────────────────────────────────────────────────────────────────
from datetime import datetime, timezone

REPO_URL  = "https://github.com/ramprasathk07/IssueFix-RL.git"
DATA_PATH = "/kaggle/input/datasets/ramprasathk07/thinking-traces/opencode_sft_filtered_sl3072_10000.jsonl"  # <-- set your dataset path

# wandb run name for THIS session — auto-timestamped so every Kaggle session
# gets a unique run without hand-editing configs/sft.yaml (or %%writefile-ing
# it — don't; see the note at the top of that file). Override the string if
# you want a specific name instead.
RUN_NAME = f"qwen0.5_sft_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M')}"

# Leave as None to use configs/sft.yaml's wandb_project; set a string to override.
WANDB_PROJECT = None
# ──────────────────────────────────────────────────────────────────────────────

In [4]:
import os, subprocess, sys

REPO_DIR = "/kaggle/working/IssueFix-RL"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
print("Repo:", os.getcwd())

# Confirm 2 GPUs
result = subprocess.run(["nvidia-smi", "--list-gpus"], capture_output=True, text=True)
gpu_lines = [l for l in result.stdout.strip().splitlines() if l.strip()]
print(f"GPUs available: {len(gpu_lines)}")
for g in gpu_lines:
    print(" ", g)

Cloning into '/kaggle/working/IssueFix-RL'...


Repo: /kaggle/working/IssueFix-RL
GPUs available: 2
  GPU 0: Tesla T4 (UUID: GPU-b87f1288-4337-1b93-5715-59154059ef9e)
  GPU 1: Tesla T4 (UUID: GPU-7a2aab9d-2cfa-8ae8-8032-dd47c09638ed)


In [5]:
# torch pre-installed on Kaggle — installs everything else
try:
    !pip install -q -r requirements.txt
except:
    pass

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 815.5 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 38.4 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 58.6 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 97.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 58.0 MB/s eta 0:00:0000:01:00:01


In [6]:
# Pull wandb key from Kaggle Secrets (set via Add-ons → Secrets in the notebook editor)
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    wandb_key = secrets.get_secret("WANDB_API_KEY")
    os.environ["WANDB_API_KEY"] = wandb_key
    print("wandb key loaded from Kaggle Secrets")
except Exception:
    print("No WANDB_API_KEY secret found — wandb logging disabled")
    os.environ["WANDB_DISABLED"] = "true"

wandb key loaded from Kaggle Secrets


In [7]:
import json
from pathlib import Path

data_file = Path(DATA_PATH)
assert data_file.exists(), (
    f"Data not found at {DATA_PATH}.\n"
    "Attach your dataset in the notebook editor: Data → Add Data, "
    "then update DATA_PATH in cell 2."
)

with open(data_file) as f:
    lines = f.readlines()
sample = json.loads(lines[0])
print(f"Examples : {len(lines)}")
print(f"Keys     : {list(sample.keys())}")
print(f"Prompt   : {sample.get('prompt','')[:100]}")
print(f"Response : {sample.get('response','')[:100]}")

Examples : 10000
Keys     : ['prompt', 'response', 'solution', 'difficulty', 'source']
Prompt   : Problem description.
Vipul is a hardworking super-hero who maintains the bracket ratio of all the st
Response : <think>
Okay, I need to solve this problem where I have to check if the brackets in a string are bal


In [8]:
# %%writefile /kaggle/working/IssueFix-RL/configs/sft.yaml
# # config_sft.yaml — Kaggle 2x T4 (15GB each)
# # Full SFT (NO LoRA)

# # ========== Model Parameters ==========
# model_params:
#   base_model: "Qwen/Qwen2.5-0.5B-Instruct"
#   model_type: "AutoModelForCausalLM"
#   trust_remote_code: true

#   # DDP-compatible settings
#   load_in_8bit: false
#   load_in_4bit: false

#   bnb_4bit_compute_dtype: "bfloat16"
#   bnb_4bit_quant_type: "nf4"
#   bnb_4bit_use_double_quant: true

#   # Full fine-tuning
#   use_lora: false


# # ========== DataLoader Parameters ==========
# dataloader_params:
#   batch_size: 1              # per GPU
#   max_length: 4096           # 6000 is too aggressive for T4s
#   shuffle: true

#   # Kaggle CPU limits
#   num_workers: 1
#   prefetch_factor: 2
#   pin_memory: true
#   pre_tokenize: true


# # ========== Training Hyperparameters ==========
# training_params:
#   # Optimization
#   learning_rate: 2e-5
#   lr_scheduler: "cosine"
#   warmup_steps: 100
#   optimizer: "adamw_torch"
#   weight_decay: 0.01
#   max_grad_norm: 1.0

#   # Training loop
#   num_epochs: 3
#   gradient_accumulation_steps: 8
#   num_gpus: 2

#   # Precision
#   bf16: true
#   fp16: false
#   tf32: true

#   # Memory savings
#   gradient_checkpointing: true

#   # Logging & checkpointing
#   output_dir: "./outputs/sft_run1"

#   logging_steps: 10
#   save_steps: 200
#   eval_steps: 200
#   save_best_k: 2

#   wandb_project: "my_sft_project_v3"
#   wandb_run_name: "qwen0.5_sft_10k_sl4096_fullft"

#   mlflow_tracking_uri: null
#   mlflow_experiment: "sft_training"

#   # Kaggle Hub upload
#   push_to_kaggle: true
#   kaggle_model_handle: "ramprasathk07/issuefix-sft/transformers/qwen0.5-sft"
#   kaggle_model_license: "apache-2.0"

In [ ]:
import yaml

with open("configs/sft.yaml") as f:
    cfg = yaml.safe_load(f)

tp = cfg["training_params"]
dp = cfg["dataloader_params"]
mp = cfg["model_params"]

# effective values — RUN_NAME/WANDB_PROJECT (cell 2) override the yaml at launch time,
# they are NOT written back to configs/sft.yaml on disk
effective_project  = WANDB_PROJECT or tp.get("wandb_project")
effective_run_name = RUN_NAME or tp.get("wandb_run_name")

eff_batch = dp["batch_size"] * tp["num_gpus"] * tp["gradient_accumulation_steps"]
print(f"Model           : {mp['base_model']}")
print(f"Epochs          : {tp['num_epochs']}")
print(f"LR              : {tp['learning_rate']}")
print(f"GPUs            : {tp['num_gpus']}")
print(f"Per-GPU batch   : {dp['batch_size']}")
print(f"Grad accum      : {tp['gradient_accumulation_steps']}")
print(f"Effective batch : {eff_batch}")
print(f"Max length      : {dp['max_length']}")
print(f"bf16            : {tp['bf16']}")
print(f"grad_ckpt       : {tp['gradient_checkpointing']}")
print(f"Output dir      : {tp['output_dir']}")
print(f"wandb project   : {effective_project}" + (" (override)" if WANDB_PROJECT else " (from yaml)"))
print(f"wandb run name  : {effective_run_name}" + (" (override)" if RUN_NAME else " (from yaml)"))
print(f"Kaggle push     : {tp.get('push_to_kaggle')} → {tp.get('kaggle_model_handle')}")

In [ ]:
# ── TRAINING ──────────────────────────────────────────────────────────────────
# notebook_launcher (called inside train.py) spawns both T4 processes.
# After all epochs: best checkpoint zipped + pushed to Kaggle Models.
# This cell is blocking — use Save Version → Save & Run All to run in background.
# RUN_NAME/WANDB_PROJECT (cell 2) override configs/sft.yaml's wandb fields for
# this run only — the yaml on disk is never touched.

extra_args = f"--wandb_run_name {RUN_NAME}"
if WANDB_PROJECT:
    extra_args += f" --wandb_project {WANDB_PROJECT}"

!python train.py --config configs/sft.yaml --data {DATA_PATH} {extra_args}

In [ ]:
# ── RESUME FROM CHECKPOINT (skip if running fresh) ────────────────────────────
# Uncomment, set CKPT, and run this cell instead of the training cell above.
# Reuses RUN_NAME/WANDB_PROJECT from cell 2 — set RUN_NAME to the ORIGINAL run's
# name if you want the resumed steps to land in the same wandb run.

# CKPT = "outputs/sft_run1/checkpoint-epoch1-step150"
# !python train.py --config configs/sft.yaml --data {DATA_PATH} --resume {CKPT} {extra_args}

In [ ]:
# ── OUTPUT SUMMARY ────────────────────────────────────────────────────────────
from pathlib import Path
import yaml

with open("configs/sft.yaml") as f:
    out_dir = Path(yaml.safe_load(f)["training_params"]["output_dir"])

checkpoints = sorted(out_dir.glob("checkpoint-*")) if out_dir.exists() else []
print(f"Checkpoints saved: {len(checkpoints)}")
for c in checkpoints:
    files = [f.name for f in c.iterdir()]
    print(f"  {c.name}: {files}")

print()
zips = list(Path("/kaggle/working").glob("*.zip"))
print(f"Zipped checkpoints ({len(zips)}):")
for z in zips:
    print(f"  {z.name}  {z.stat().st_size / 1e6:.1f} MB")

In [ ]:
# ── CHECKPOINT TESTS + SAMPLE GENERATIONS (optional) ─────────────────────────
checkpoints = sorted(Path(out_dir).glob("checkpoint-*")) if out_dir.exists() else []
if checkpoints:
    latest = checkpoints[-1]
    print(f"Testing checkpoint: {latest}")
    !pytest src/tests/test_checkpoint.py -v -s --ckpt {latest}
else:
    print("No checkpoint found — training may have failed")

In [ ]:
# ── PUBLISH TO KAGGLE MODELS (date + project title + final loss in notes) ────
# Training already auto-pushes if push_to_kaggle: true in the config, but with
# generic version-notes ("Best checkpoint — run: ..."). This cell re-publishes
# the latest checkpoint with richer notes — safe to run standalone/idempotently.

import subprocess
from datetime import datetime, timezone
from pathlib import Path
import yaml

with open("configs/sft.yaml") as f:
    cfg = yaml.safe_load(f)
tp = cfg["training_params"]

# use the RUN_NAME/WANDB_PROJECT this session actually trained with (cell 2/10),
# not whatever is sitting in the yaml on disk — falls back to the yaml value
# only if cell 2 wasn't run in this kernel session (e.g. after a restart)
run_name = RUN_NAME if "RUN_NAME" in dir() else tp["wandb_run_name"]
project = (WANDB_PROJECT if "WANDB_PROJECT" in dir() and WANDB_PROJECT else None) or tp["wandb_project"]

out_dir = Path(tp["output_dir"])
checkpoints = sorted(out_dir.glob("checkpoint-*"), key=lambda p: p.stat().st_mtime) if out_dir.exists() else []
assert checkpoints, f"No checkpoints found under {out_dir} — train first."
latest_ckpt = checkpoints[-1]
print(f"Checkpoint: {latest_ckpt}")

# Pull the final loss from the wandb run that matches this session's run name
final_loss = "n/a"
try:
    import wandb
    api = wandb.Api()
    entity = api.default_entity
    matches = [r for r in api.runs(f"{entity}/{project}") if r.name == run_name]
    if matches:
        run = sorted(matches, key=lambda r: r.created_at)[-1]  # most recent matching run
        loss = run.summary.get("val/loss", run.summary.get("train/loss"))
        if loss is not None:
            final_loss = f"{loss:.4f}"
    print(f"Final loss (wandb): {final_loss}")
except Exception as e:
    print(f"Could not fetch loss from wandb ({e}) — publishing without it")

date_str = datetime.now(timezone.utc).strftime("%Y-%m-%d")
project_title = "IssueFix-RL SFT"
version_notes = f"{project_title} | {date_str} | run: {run_name} | final loss: {final_loss}"
print(f"Version notes: {version_notes}")

handle = tp["kaggle_model_handle"]  # "owner/model/framework/variation"
owner, model_slug, framework, variation = handle.split("/")

subprocess.run(
    ["kaggle", "models", "create",
     "--owner", owner, "--name", model_slug,
     "--framework", framework,
     "--license", tp.get("kaggle_model_license", "apache-2.0")],
    capture_output=True,
)  # no-op if the model already exists

result = subprocess.run(
    ["kaggle", "models", "instances", "versions", "create", handle,
     "--path", str(latest_ckpt),
     "--version-notes", version_notes],
    capture_output=True, text=True,
)
if result.returncode == 0:
    print(f"Pushed -> https://www.kaggle.com/models/{owner}/{model_slug}")
else:
    print(f"Push failed:\n{result.stderr.strip()}")